# 01 - Data Audit

Phase 0 audit of the raw WHO/World Bank extracts used in this project,
before any cleaning is applied. See `data/README.md` for the provenance of
each file (3 real, 2 simulated).

In [1]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src import data_loader

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

raw = data_loader.load_all()
for name, df in raw.items():
    print(f"{name:22s} shape={df.shape}")

coverage               shape=(37749, 9)
reported_cases         shape=(64860, 7)
population             shape=(9710, 3)
vaccine_introduction   shape=(1565, 6)
vaccine_schedule       shape=(1824, 12)
country_reference      shape=(234, 3)


## Coverage

In [2]:
cov = raw["coverage"]
cov.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37749 entries, 0 to 37748
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Group                          37749 non-null  object 
 1   Code                           37749 non-null  object 
 2   Name                           37749 non-null  object 
 3   Year                           37749 non-null  int64  
 4   Antigen                        37749 non-null  object 
 5   Antigen_description            37749 non-null  object 
 6   Coverage                       37749 non-null  float64
 7   Coverage_category              37749 non-null  object 
 8   Coverage_category_description  37749 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 2.6+ MB


In [3]:
cov.head()

,Group,Code,Name,Year,Antigen,Antigen_description,Coverage,Coverage_category,Coverage_category_description
0,COUNTRIES,PAK,Pakistan,2015,BCG,Bacille Calmette-Guerin vaccine (tuberculosis),86.0,WUENIC,WHO/UNICEF Estimates of National Immunization ...
1,COUNTRIES,ARE,United Arab Emirates,2024,BCG,Bacille Calmette-Guerin vaccine (tuberculosis),96.0,WUENIC,WHO/UNICEF Estimates of National Immunization ...
2,COUNTRIES,UKR,Ukraine,2003,BCG,Bacille Calmette-Guerin vaccine (tuberculosis),98.0,WUENIC,WHO/UNICEF Estimates of National Immunization ...
3,COUNTRIES,POL,Poland,2012,BCG,Bacille Calmette-Guerin vaccine (tuberculosis),94.0,WUENIC,WHO/UNICEF Estimates of National Immunization ...
4,COUNTRIES,PNG,Papua New Guinea,2005,BCG,Bacille Calmette-Guerin vaccine (tuberculosis),90.0,WUENIC,WHO/UNICEF Estimates of National Immunization ...


In [4]:
print("Year range:", cov["Year"].min(), "-", cov["Year"].max())
print("Antigens:", sorted(cov["Antigen"].unique()))
print("Coverage_category:", sorted(cov["Coverage_category"].unique()))
print("Coverage range:", cov["Coverage"].min(), "-", cov["Coverage"].max())
print("Countries:", cov["Code"].nunique())
print("Duplicate rows (Code, Year, Antigen, Coverage_category):",
      cov.duplicated(subset=["Code", "Year", "Antigen", "Coverage_category"]).sum())
print("Missing values per column:")
print(cov.isna().sum())

Year range: 2000 - 2025
Antigens: ['BCG', 'DTP3', 'HEPB3', 'HIB3', 'IPV1', 'MCV1', 'MCV2', 'PCV3', 'ROTAC']
Coverage_category: ['WUENIC']
Coverage range: 0.0 - 99.0
Countries: 195
Duplicate rows (Code, Year, Antigen, Coverage_category): 0
Missing values per column:
Group                            0
Code                             0
Name                             0
Year                             0
Antigen                          0
Antigen_description              0
Coverage                         0
Coverage_category                0
Coverage_category_description    0
dtype: int64


## Reported cases

In [5]:
cases = raw["reported_cases"]
cases.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64860 entries, 0 to 64859
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Group                64860 non-null  object 
 1   Code                 64860 non-null  object 
 2   Name                 64860 non-null  object 
 3   Year                 64860 non-null  int64  
 4   Disease              64860 non-null  object 
 5   Disease_description  64860 non-null  object 
 6   Cases                54725 non-null  float64
dtypes: float64(1), int64(1), object(5)
memory usage: 3.5+ MB


In [6]:
print("Year range:", cases["Year"].min(), "-", cases["Year"].max())
print("Diseases:", sorted(cases["Disease"].unique()))
print("Countries:", cases["Code"].nunique())
print("Duplicate rows (Code, Year, Disease):",
      cases.duplicated(subset=["Code", "Year", "Disease"]).sum())
print("Missing Cases:", cases["Cases"].isna().sum(), "/", len(cases),
      f"({cases['Cases'].isna().mean():.1%})")
print("Negative Cases:", (cases["Cases"] < 0).sum())

Year range: 1974 - 2025
Diseases: ['DIPHTHERIA', 'MEASLES', 'PERTUSSIS', 'POLIO', 'RUBELLA', 'TETANUS_NEONATAL', 'TETANUS_TOTAL']
Countries: 214
Duplicate rows (Code, Year, Disease): 0
Missing Cases: 10135 / 64860 (15.6%)
Negative Cases: 0


## Population (auxiliary, used to derive incidence rate)

In [7]:
pop = raw["population"]
print(pop.shape)
print("Year range:", pop["Year"].min(), "-", pop["Year"].max())
print("Countries:", pop["Code"].nunique())
pop.describe()

(9710, 3)
Year range: 1980 - 2024
Countries: 216


,Year,Population
count,9710.000000,9.710000e+03
mean,2002.018023,2.918063e+07
std,12.982046,1.182080e+08
min,1980.000000,7.366000e+03
25%,1991.000000,6.239102e+05
50%,2002.000000,5.116400e+06
75%,2013.000000,1.747840e+07
max,2024.000000,1.450936e+09


## Vaccine introduction (SIMULATED)

Real country/ISO3/WHO-region and real vaccine names; introduction years
and Yes/No status are seeded-random. See `data/README.md`.

In [8]:
intro = raw["vaccine_introduction"]
intro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1565 entries, 0 to 1564
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ISO_3_Code    1565 non-null   object 
 1   Country_Name  1565 non-null   object 
 2   WHO_Region    1565 non-null   object 
 3   Year          1128 non-null   float64
 4   Description   1565 non-null   object 
 5   Intro         1565 non-null   object 
dtypes: float64(1), object(5)
memory usage: 73.5+ KB


In [9]:
print("Vaccines:", sorted(intro["Description"].unique()))
print("WHO regions:", sorted(intro["WHO_Region"].unique()))
print("Intro value counts:")
print(intro["Intro"].value_counts())
print("Countries:", intro["ISO_3_Code"].nunique())

Vaccines: ['Hepatitis B birth dose', 'Human papillomavirus vaccine', 'Inactivated polio vaccine', 'Japanese encephalitis vaccine', 'Meningococcal A conjugate vaccine', 'Pneumococcal conjugate vaccine', 'Rotavirus vaccine', 'Rubella-containing vaccine, 1st dose', 'Yellow fever vaccine']
WHO regions: ['AFR', 'AMR', 'EMR', 'EUR', 'SEAR', 'WPR']
Intro value counts:
Intro
Yes    1128
No      437
Name: count, dtype: int64
Countries: 228


## Vaccine schedule (SIMULATED)

Real country/ISO3/WHO-region and real vaccine names; schedule rounds and
ages are seeded-random. See `data/README.md`.

In [10]:
sched = raw["vaccine_schedule"]
sched.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1824 entries, 0 to 1823
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   ISO_3_Code              1824 non-null   object
 1   Country_Name            1824 non-null   object
 2   WHO_Region              1824 non-null   object
 3   Year                    1824 non-null   int64 
 4   Vaccine_code            1824 non-null   object
 5   Vaccine_description     1824 non-null   object
 6   Schedule_rounds         1824 non-null   int64 
 7   Target_pop              1824 non-null   object
 8   Target_pop_description  1824 non-null   object
 9   Geoarea                 1824 non-null   object
 10  Age_administered        1824 non-null   object
 11  Source_comment          1824 non-null   object
dtypes: int64(2), object(10)
memory usage: 171.1+ KB


In [11]:
print("Vaccine codes:", sorted(sched["Vaccine_code"].unique()))
print("Target populations:", sorted(sched["Target_pop"].unique()))
print("Schedule_rounds range:", sched["Schedule_rounds"].min(), "-", sched["Schedule_rounds"].max())
print("Countries:", sched["ISO_3_Code"].nunique())

Vaccine codes: ['BCG', 'DTP', 'HEPB', 'HIB', 'MCV', 'OPV', 'PCV', 'ROTA']
Target populations: ['INFANTS', 'INFANTS;CHILDREN']
Schedule_rounds range: 1 - 4
Countries: 228


## Country reference

In [12]:
ref = raw["country_reference"]
print(ref.shape)
print(ref["WHO_Region"].value_counts(dropna=False))

(234, 3)
WHO_Region
EUR     58
AMR     52
AFR     50
WPR     35
EMR     23
SEAR    10
NaN      6
Name: count, dtype: int64


## Audit conclusions

- Coverage and reported-cases are well-formed long-format tables already
  consistent with the schema described in the project brief: no out-of-
  range percentages, no negative case counts, no duplicate keys.
- Reported cases has a real (not injected) 15.6% missing rate on `Cases`,
  reflecting genuine WHO reporting gaps -- this drives a cleaning decision
  documented in `reports/data_quality_report.md` (retain as null, do not
  impute zero).
- Vaccine introduction and vaccine schedule are simulated placeholders for
  data WHO does not expose through a public bulk API; they are structurally
  valid (correct schema, real countries/regions/vaccines) but their
  numeric content should not be read as real WHO findings.
- No demographic (gender, education, income), sub-national, or sub-annual
  fields exist anywhere in the source data. This rules out several brief
  questions outright -- see `docs/limitations.md`.